In [1]:
# Day 2 - Groq API Setup

!pip install -q openai

from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-20b"

print("Groq API connected successfully!")

Groq API connected successfully!


In [2]:
# Task 1: Understanding Messages and Roles

messages = [
    {
        "role": "system",
        "content": "You are a friendly assistant for Sweet Crumbs Home Bakery."
    },
    {
        "role": "user",
        "content": "What is the price of a chocolate cake?"
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages
)

print("Customer:", messages[1]["content"])
print("Assistant:", response.choices[0].message.content)

Customer: What is the price of a chocolate cake?
Assistant: Hi there!  
Our classic 8‑inch chocolate cake is priced at **$30**.  
If you’re looking for a larger size, a 10‑inch cake comes to **$45**, and a 12‑inch (party) cake is **$60**.  

Let me know which size you’d like, or if you have any special requests (e.g., custom frosting, gluten‑free, or vegan). We’re happy to help make your treat perfect!


In [3]:
# Task 2: System Prompt - Role, Facts, Rules, Format

messages = [
    {
        "role": "system",
        "content": """
        ROLE: You are a friendly assistant for Sweet Crumbs Home Bakery.

        FACTS:
        - Chocolate cake costs Rs. 650.
        - Plum cake costs Rs. 500.
        - All cakes are available in eggless versions.

        RULES:
        - Always answer using the given facts.
        - Never invent prices or information.
        - If information is unavailable, politely say you don't know.
        - Always be polite and helpful.

        FORMAT:
        - Give short and clear answers.
        """
    },
    {
        "role": "user",
        "content": "What is the price of a chocolate cake?"
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0.2
)

print("Customer:", messages[1]["content"])
print("Assistant:", response.choices[0].message.content)

Customer: What is the price of a chocolate cake?
Assistant: Chocolate cake costs Rs. 650.


In [4]:
# Task 3: Few-Shot Examples

messages = [
    {
        "role": "system",
        "content": """
        You are a customer support assistant for Sweet Crumbs Home Bakery.

        Follow the style of these examples:

        Customer: What is the price of plum cake?
        Assistant: Our plum cake costs Rs. 500.

        Customer: Do you have eggless cakes?
        Assistant: Yes! All our cakes are available in eggless versions.

        Customer: How much is chocolate cake?
        Assistant: Our chocolate cake costs Rs. 650.

        Answer new customer questions in the same style.
        Do not invent information.
        """
    },
    {
        "role": "user",
        "content": "Do you deliver to Kakinada?"
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0.2
)

print("Customer:", messages[1]["content"])
print("Assistant:", response.choices[0].message.content)

Customer: Do you deliver to Kakinada?
Assistant: I’m sorry, I don’t have that information.


In [5]:
# Task 4: JSON Output

messages = [
    {
        "role": "system",
        "content": """
        You are a Sweet Crumbs Home Bakery assistant.

        Answer customer questions using JSON format only.

        JSON must contain:
        - customer_question
        - answer
        - category

        Use these facts:
        - Chocolate cake costs Rs. 650.
        - Plum cake costs Rs. 500.
        - All cakes have eggless versions.

        Never invent information.
        """
    },
    {
        "role": "user",
        "content": "What is the price of chocolate cake?"
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0.2
)

print(response.choices[0].message.content)

{"customer_question":"What is the price of chocolate cake?","answer":"The chocolate cake costs Rs. 650.","category":"pricing"}


In [6]:
# Task 5: Conversation Memory

messages = [
    {
        "role": "system",
        "content": """
        You are a friendly assistant for Sweet Crumbs Home Bakery.
        Remember details shared by the customer during this conversation.
        """
    },
    {
        "role": "user",
        "content": "My name is Ravi."
    },
    {
        "role": "assistant",
        "content": "Hello Ravi! How can I help you today?"
    },
    {
        "role": "user",
        "content": "What is my name?"
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0.2
)

print("Customer:", messages[-1]["content"])
print("Assistant:", response.choices[0].message.content)

Customer: What is my name?
Assistant: You’re Ravi! If there’s anything else you’d like to chat about—maybe a recipe or a special order—just let me know.


In [7]:
# Task 6: Python Dictionaries and JSON

import json

customer_order = {
    "customer_name": "Ravi",
    "cake_type": "Chocolate Cake",
    "quantity": 1,
    "price": 650,
    "eggless": True
}

# Convert Python dictionary into JSON
order_json = json.dumps(customer_order, indent=4)

print("Customer Order:")
print(order_json)

# Convert JSON back into Python dictionary
order_data = json.loads(order_json)

print("\nCustomer Name:", order_data["customer_name"])
print("Cake Type:", order_data["cake_type"])
print("Total Price: Rs.", order_data["price"])

Customer Order:
{
    "customer_name": "Ravi",
    "cake_type": "Chocolate Cake",
    "quantity": 1,
    "price": 650,
    "eggless": true
}

Customer Name: Ravi
Cake Type: Chocolate Cake
Total Price: Rs. 650


In [8]:
# Task 7: Functions and Type Hints

def calculate_total(price: int, quantity: int) -> int:
    """Calculate the total price of a cake order."""
    return price * quantity


# Customer order
cake_price: int = 650
quantity: int = 2

total = calculate_total(cake_price, quantity)

print("Cake Price: Rs.", cake_price)
print("Quantity:", quantity)
print("Total Amount: Rs.", total)

Cake Price: Rs. 650
Quantity: 2
Total Amount: Rs. 1300


In [9]:
# Task 8: Decorators

def order_logger(func):
    def wrapper():
        print("Processing customer order...")
        func()
        print("Order processing completed!")
    return wrapper


@order_logger
def place_order():
    print("Chocolate cake order placed successfully.")


place_order()

Processing customer order...
Chocolate cake order placed successfully.
Order processing completed!


In [10]:
# Task 9: Try and Except Error Handling

def calculate_order(price, quantity):
    try:
        total = price * int(quantity)
        print("Total Amount: Rs.", total)

    except ValueError:
        print("Invalid quantity! Please enter a valid number.")

    except Exception as error:
        print("Something went wrong:", error)


# Test 1: Valid quantity
print("Test 1:")
calculate_order(650, 2)

# Test 2: Invalid quantity
print("\nTest 2:")
calculate_order(650, "two")

Test 1:
Total Amount: Rs. 1300

Test 2:
Invalid quantity! Please enter a valid number.


In [11]:
customer_questions = [
    "What is the price of a chocolate cake?",
    "What is the price of a plum cake?",
    "Do you have eggless cakes?",
    "Do you provide home delivery?",
    "Do you deliver to Kakinada?",
    "How many days in advance should I order?",
    "What flavours are available?",
    "Can I customize my cake?",
    "What payment methods do you accept?",
    "How can I place an order?"
]

for number, question in enumerate(customer_questions, start=1):
    print(f"{number}. {question}")

1. What is the price of a chocolate cake?
2. What is the price of a plum cake?
3. Do you have eggless cakes?
4. Do you provide home delivery?
5. Do you deliver to Kakinada?
6. How many days in advance should I order?
7. What flavours are available?
8. Can I customize my cake?
9. What payment methods do you accept?
10. How can I place an order?


In [12]:
business_catalogue = {
    "business_name": "Sweet Crumbs Home Bakery",
    "products": {
        "Chocolate Cake": {
            "price": 650,
            "eggless_available": True
        },
        "Plum Cake": {
            "price": 500,
            "eggless_available": True
        }
    },
    "delivery": {
        "home_delivery": "Information not available",
        "service_areas": "Information not available"
    },
    "ordering": {
        "advance_booking": "Information not available",
        "customization": "Information not available"
    }
}

print(business_catalogue)

{'business_name': 'Sweet Crumbs Home Bakery', 'products': {'Chocolate Cake': {'price': 650, 'eggless_available': True}, 'Plum Cake': {'price': 500, 'eggless_available': True}}, 'delivery': {'home_delivery': 'Information not available', 'service_areas': 'Information not available'}, 'ordering': {'advance_booking': 'Information not available', 'customization': 'Information not available'}}


In [13]:
# Day 2 Final Task: Sweet Crumbs AI Chatbot

import json

# Business Catalogue
business_catalogue = {
    "business_name": "Sweet Crumbs Home Bakery",
    "products": {
        "Chocolate Cake": {
            "price": 650,
            "eggless_available": True
        },
        "Plum Cake": {
            "price": 500,
            "eggless_available": True
        }
    }
}

# Conversation Memory
chat_history = []

# Calculate Order Price
def calculate_total(price: int, quantity: int) -> int:
    return price * quantity


# Process Customer Order
def process_order(customer_name, cake_type, quantity, eggless):

    try:
        quantity = int(quantity)

        if quantity <= 0:
            print("Please enter a valid quantity.")
            return

        if cake_type not in business_catalogue["products"]:
            print("Sorry, this cake is not available.")
            return

        price = business_catalogue["products"][cake_type]["price"]

        total = calculate_total(price, quantity)

        order = {
            "customer_name": customer_name,
            "cake_type": cake_type,
            "quantity": quantity,
            "price": price,
            "total_amount": total,
            "eggless": eggless
        }

        print("\nProcessing customer order...")
        print(json.dumps(order, indent=4))
        print("Order placed successfully!")

    except ValueError:
        print("Invalid quantity! Please enter a number.")

    except Exception as error:
        print("Something went wrong:", error)


# AI Chatbot
def bakery_chatbot(customer_message):

    chat_history.append({
        "role": "user",
        "content": customer_message
    })

    messages = [
        {
            "role": "system",
            "content": """
            You are a friendly assistant for Sweet Crumbs Home Bakery.

            Business Facts:
            - Chocolate Cake costs Rs. 650.
            - Plum Cake costs Rs. 500.
            - All cakes are available in eggless versions.

            Rules:
            - Remember the conversation history.
            - Never invent prices or business information.
            - If information is unavailable, say so politely.
            - Give short and helpful answers.
            """
        }
    ] + chat_history

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2
    )

    answer = response.choices[0].message.content

    chat_history.append({
        "role": "assistant",
        "content": answer
    })

    return answer


# Test 1: Customer Questions
print("Customer:", "What is the price of chocolate cake?")
print("Assistant:", bakery_chatbot(
    "What is the price of chocolate cake?"
))

# Test 2: Conversation Memory
print("\nCustomer:", "My name is Ravi.")
print("Assistant:", bakery_chatbot("My name is Ravi."))

print("\nCustomer:", "What is my name?")
print("Assistant:", bakery_chatbot("What is my name?"))

# Test 3: Order Processing
print("\nCustomer Order:")
process_order("Ravi", "Chocolate Cake", 2, True)

Customer: What is the price of chocolate cake?
Assistant: Chocolate cake is Rs. 650.

Customer: My name is Ravi.
Assistant: Nice to meet you, Ravi! How can I help you today?

Customer: What is my name?
Assistant: Your name is Ravi.

Customer Order:

Processing customer order...
{
    "customer_name": "Ravi",
    "cake_type": "Chocolate Cake",
    "quantity": 2,
    "price": 650,
    "total_amount": 1300,
    "eggless": true
}
Order placed successfully!
